# Stack Ensemble —> XGBoost como Meta-Modelo

**stack ensemble** usando **XGBoost** como meta-modelo, para comparar com a versao que usa Regressao Logistica.

A ideia e a mesma: o meta-modelo aprende a combinar as probabilidades dos 5 modelos base (SVM, KNN, Extra Trees, Regressao Logistica, LightGBM). A diferenca e que o XGBoost pode capturar **relacoes nao-lineares** entre as probabilidades, o que pode (ou nao) melhorar os resultados.

### Como funciona:

Para cada `Janela Incremental` e cada `Temporada`, o meta-modelo aprende **progressivamente** dentro da propria temporada:

- **Janela = 5**: treina com as linhas 1 a 5, prediz a 6. Depois treina com 1 a 6, prediz a 7. E assim por diante.
- **Janela = 10**: treina com as linhas 1 a 10, prediz a 11.
- **Janela = 15**: treina com as linhas 1 a 15, prediz a 16.

### Features do meta-modelo:
As 5 probabilidades dos modelos base: `prob_svm`, `prob_knn`, `prob_extra_trees`, `prob_regressao_logistica`, `prob_lightgbm`

### Target:
`Resultado Real` (0 ou 1)

## Importar bibliotecas

In [ ]:
import os
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, f1_score
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick

## Configuracao

- `META_DATASET_PATH`: caminho do meta-dataset com as probabilidades dos 5 modelos
- `SAIDA_STACK`: pasta onde salvarei os resultados do stack ensemble com XGBoost
- `SAIDA_EXP03`: pasta dos resultados do experimento 03 (para comparacao nos graficos)
- `FEATURES`: as 5 colunas de probabilidade que o meta-modelo usa como entrada
- `TARGET`: a coluna alvo

Para o XGBoost, usei parametros conservadores para evitar overfitting nas janelas pequenas:
- `max_depth=3`: arvores rasas para nao decorar os dados
- `n_estimators=50`: poucas arvores ja que o treino comeca com apenas 5-15 amostras
- `learning_rate=0.1`: taxa de aprendizado padrao

In [ ]:
META_DATASET_PATH = os.path.abspath(
    os.path.join('..', 'results', 'experimento_04_stack', 'meta_dataset_stack.csv')
)
SAIDA_STACK = os.path.abspath(
    os.path.join('..', 'results', 'experimento_04_stack')
)
SAIDA_EXP03 = os.path.abspath(
    os.path.join('..', 'results', 'experimento_03')
)

FEATURES = [
    'prob_svm',
    'prob_knn',
    'prob_extra_trees',
    'prob_regressao_logistica',
    'prob_lightgbm',
]

TARGET = 'Resultado Real'

JANELAS = [5, 10, 15]

print(f'Meta-dataset: {META_DATASET_PATH}')
print(f'Saida: {SAIDA_STACK}')
print(f'Features: {FEATURES}')
print(f'Target: {TARGET}')

## Carregar o meta-dataset

Carreguei o mesmo meta-dataset usado no stack ensemble com Regressao Logistica. Cada linha representa um jogo, com as probabilidades previstas pelos 5 modelos base e o resultado real.

In [ ]:
df_meta = pd.read_csv(META_DATASET_PATH)

print(f'Shape: {df_meta.shape}')
print(f'Janelas: {sorted(df_meta["Janela Incremental"].unique())}')
print(f'Temporadas: {sorted(df_meta["Temporada"].unique())}')
df_meta.head()

## Implementacao do Stack Ensemble com XGBoost

Aqui implementei a mesma logica de janela incremental, mas usando **XGBoost** como meta-learner:

1. Para cada `Janela Incremental` e cada `Temporada`, filtrei os dados e ordenei pela `Ordem Jogo Temporada`
2. Comecei treinando com as primeiras `janela` linhas e prevendo a proxima
3. A cada iteracao, expandi o conjunto de treino com mais uma linha e previ a seguinte
4. Usei **XGBClassifier** com parametros conservadores para evitar overfitting
5. Armazenei cada previsao para calcular as metricas depois

**Obs:** Assim como na versao com Regressao Logistica, quando o `y_train` tem apenas uma classe, previ diretamente essa classe.

In [ ]:
todas_predicoes = []
resultados_resumo = []

for janela in JANELAS:
    df_janela = df_meta[df_meta['Janela Incremental'] == janela].copy()
    temporadas = sorted(df_janela['Temporada'].unique())

    for temporada in temporadas:
        df_temp = df_janela[df_janela['Temporada'] == temporada].copy()
        df_temp = df_temp.sort_values('Ordem Jogo Temporada').reset_index(drop=True)

        n_jogos = len(df_temp)

        if n_jogos <= janela:
            print(
                f'[AVISO] Janela={janela} | {temporada} | '
                f'Apenas {n_jogos} jogos, insuficiente para treinar e prever. Pulando.'
            )
            continue

        y_true_temp = []
        y_pred_temp = []
        y_prob_temp = []

        for i in range(janela, n_jogos):
            X_train = df_temp.loc[:i-1, FEATURES]
            y_train = df_temp.loc[:i-1, TARGET]

            X_test = df_temp.loc[[i], FEATURES]
            y_test = df_temp.loc[i, TARGET]

            if y_train.nunique() < 2:
                classe_unica = int(y_train.iloc[0])
                pred = classe_unica
                prob_classe_1 = 1.0 if classe_unica == 1 else 0.0
            else:
                meta_model = XGBClassifier(
                    n_estimators=50,
                    max_depth=3,
                    learning_rate=0.1,
                    objective='binary:logistic',
                    eval_metric='logloss',
                    use_label_encoder=False,
                    verbosity=0,
                    random_state=42
                )
                meta_model.fit(X_train, y_train)

                pred = meta_model.predict(X_test)[0]
                prob = meta_model.predict_proba(X_test)[0]
                prob_classe_1 = float(prob[1]) if len(prob) > 1 else float(prob[0])

            y_true_temp.append(int(y_test))
            y_pred_temp.append(int(pred))
            y_prob_temp.append(prob_classe_1)

            todas_predicoes.append({
                'Janela Incremental': janela,
                'Temporada': temporada,
                'Ordem Jogo Temporada': int(df_temp.loc[i, 'Ordem Jogo Temporada']),
                'Resultado Real': int(y_test),
                'Previsao': int(pred),
                'Probabilidade Classe 1': prob_classe_1,
            })

        if len(y_true_temp) > 0:
            acc = accuracy_score(y_true_temp, y_pred_temp)
            f1 = f1_score(y_true_temp, y_pred_temp, average='weighted', zero_division=0)

            resultados_resumo.append({
                'Janela Incremental': janela,
                'Temporada': temporada,
                'Jogos Avaliados': len(y_true_temp),
                'Acur\u00e1cia': round(acc, 4),
                'F1-Score': round(f1, 4),
            })

            print(
                f'Janela={janela} | {temporada} | '
                f'Jogos: {len(y_true_temp)} | '
                f'Acuracia: {acc:.4f} | '
                f'F1: {f1:.4f}'
            )

print(f'\nTotal de predicoes: {len(todas_predicoes)}')

## Salvar resultados

Salvei dois CSVs:
- **Predicoes detalhadas**: cada linha e uma previsao do stack ensemble (XGBoost) com a probabilidade
- **Resumo por temporada**: acuracia e F1-Score agregados por janela e temporada

In [ ]:
os.makedirs(SAIDA_STACK, exist_ok=True)

df_predicoes = pd.DataFrame(todas_predicoes)
predicoes_path = os.path.join(SAIDA_STACK, 'stack_ensemble_xgboost_predicoes.csv')
df_predicoes.to_csv(predicoes_path, index=False)
print(f'Predicoes salvas em: {predicoes_path}')
print(f'Shape: {df_predicoes.shape}')

df_resumo_xgb = pd.DataFrame(resultados_resumo)
resumo_path = os.path.join(SAIDA_STACK, 'stack_ensemble_xgboost_resumo.csv')
df_resumo_xgb.to_csv(resumo_path, index=False)
print(f'\nResumo salvo em: {resumo_path}')
print(df_resumo_xgb.to_string(index=False))

## Funcao de Plot

Criei a funcao `plot_comparativo(janela)` que gera um grafico comparando o stack ensemble (XGBoost) com:
- Os 5 modelos individuais
- O voto majoritario
- O soft voting
- O stack ensemble com Regressao Logistica

Assim consigo visualizar se o XGBoost traz alguma melhora em relacao a Regressao Logistica.

In [ ]:
def plot_comparativo(janela):
    modelos = {
        'SVM':                'svm',
        'KNN':                'knn',
        'Extra Trees':        'extra_trees',
        'Regressao Logistica':'regressao_logistica',
        'LightGBM':           'lightgbm',
    }

    jogos_por_temporada = {
        '2008-2009': 237, '2009-2010': 162, '2011-2012': 176,
        '2012-2013': 350, '2013-2014': 316, '2014-2015': 285,
        '2015-2016': 255, '2016-2017': 260, '2018-2019': 218,
        '2019-2020': 207, '2020-2021': 266, '2021-2022': 304,
        '2022-2023': 312, '2023-2024': 386
    }

    cores = {
        'SVM':                '#1f77b4',
        'KNN':                '#ff7f0e',
        'Extra Trees':        '#2ca02c',
        'Regressao Logistica':'#d62728',
        'LightGBM':           '#9467bd',
    }

    marcadores = {
        'SVM':                'o',
        'KNN':                's',
        'Extra Trees':        '^',
        'Regressao Logistica':'D',
        'LightGBM':           'P',
    }

    temporadas_order = list(jogos_por_temporada.keys())

    dfs_plot = []
    for nome_modelo, arquivo_modelo in modelos.items():
        path = os.path.join(
            SAIDA_EXP03,
            f'{arquivo_modelo}_experimento_03_janela{janela}.csv'
        )
        if not os.path.exists(path):
            continue
        df_m = pd.read_csv(path)
        df_m['Modelo'] = nome_modelo
        dfs_plot.append(df_m)

    dados_plot = pd.concat(dfs_plot, ignore_index=True)
    dados_plot['Acur\u00e1cia'] = dados_plot['Acur\u00e1cia'].astype(float)
    dados_plot['Temporada'] = pd.Categorical(
        dados_plot['Temporada'], categories=temporadas_order, ordered=True
    )
    dados_plot = dados_plot.sort_values('Temporada')

    ensembles = {}

    voto_path = os.path.join(SAIDA_EXP03, 'ensemble_voto_majoritario.csv')
    if os.path.exists(voto_path):
        df_voto = pd.read_csv(voto_path)
        df_voto = df_voto[df_voto['Janela Incremental'] == janela].copy()
        df_voto['Acur\u00e1cia'] = df_voto['Acur\u00e1cia'].astype(float)
        df_voto['Temporada'] = pd.Categorical(
            df_voto['Temporada'], categories=temporadas_order, ordered=True
        )
        df_voto = df_voto.sort_values('Temporada')
        ensembles['Voto Majoritario'] = df_voto

    soft_path = os.path.join(SAIDA_EXP03, 'ensemble_soft_voting.csv')
    if os.path.exists(soft_path):
        df_soft = pd.read_csv(soft_path)
        df_soft = df_soft[df_soft['Janela Incremental'] == janela].copy()
        df_soft['Acur\u00e1cia'] = df_soft['Acur\u00e1cia'].astype(float)
        df_soft['Temporada'] = pd.Categorical(
            df_soft['Temporada'], categories=temporadas_order, ordered=True
        )
        df_soft = df_soft.sort_values('Temporada')
        ensembles['Soft Voting'] = df_soft

    lr_path = os.path.join(SAIDA_STACK, 'stack_ensemble_resumo.csv')
    if os.path.exists(lr_path):
        df_lr = pd.read_csv(lr_path)
        df_lr = df_lr[df_lr['Janela Incremental'] == janela].copy()
        df_lr['Acur\u00e1cia'] = df_lr['Acur\u00e1cia'].astype(float)
        df_lr['Temporada'] = pd.Categorical(
            df_lr['Temporada'], categories=temporadas_order, ordered=True
        )
        df_lr = df_lr.sort_values('Temporada')
        ensembles['Stack (Log. Reg.)'] = df_lr

    df_stack = df_resumo_xgb[df_resumo_xgb['Janela Incremental'] == janela].copy()
    df_stack['Temporada'] = pd.Categorical(
        df_stack['Temporada'], categories=temporadas_order, ordered=True
    )
    df_stack = df_stack.sort_values('Temporada')

    fig, (ax1, ax2) = plt.subplots(
        2, 1,
        figsize=(16, 9),
        gridspec_kw={'height_ratios': [3, 1]},
        sharex=True
    )

    for nome_modelo in modelos.keys():
        subset = dados_plot[dados_plot['Modelo'] == nome_modelo].sort_values('Temporada')
        ax1.plot(
            subset['Temporada'],
            subset['Acur\u00e1cia'],
            marker=marcadores[nome_modelo],
            color=cores[nome_modelo],
            linewidth=1.5,
            markersize=5,
            label=nome_modelo,
            alpha=0.4
        )

    if 'Voto Majoritario' in ensembles:
        df_e = ensembles['Voto Majoritario']
        ax1.plot(
            df_e['Temporada'], df_e['Acur\u00e1cia'],
            marker='x', color='gray', linewidth=1.5, markersize=7,
            linestyle=':', label='Voto Majoritario', alpha=0.5
        )

    if 'Soft Voting' in ensembles:
        df_e = ensembles['Soft Voting']
        ax1.plot(
            df_e['Temporada'], df_e['Acur\u00e1cia'],
            marker='+', color='dimgray', linewidth=1.5, markersize=7,
            linestyle='-.', label='Soft Voting', alpha=0.5
        )

    if 'Stack (Log. Reg.)' in ensembles:
        df_e = ensembles['Stack (Log. Reg.)']
        ax1.plot(
            df_e['Temporada'], df_e['Acur\u00e1cia'],
            marker='d', color='blue', linewidth=2, markersize=8,
            linestyle='--', label='Stack (Log. Reg.)', alpha=0.7
        )

    ax1.plot(
        df_stack['Temporada'],
        df_stack['Acur\u00e1cia'],
        marker='*',
        color='red',
        linewidth=3,
        markersize=12,
        linestyle='--',
        label='Stack (XGBoost)',
        zorder=10
    )

    ax1.set_title(
        f'Stack Ensemble XGBoost vs Todos | Janela = {janela}',
        fontsize=14
    )
    ax1.set_ylabel('Acuracia', fontsize=12)
    ax1.set_ylim(0.45, 0.90)
    ax1.yaxis.set_major_formatter(mtick.FormatStrFormatter('%.2f'))
    ax1.legend(loc='upper right', fontsize=8, ncol=2)
    ax1.grid(axis='y', linestyle='--', alpha=0.5)
    ax1.tick_params(axis='x', rotation=45)

    temporadas_presentes = [t for t in temporadas_order if t in jogos_por_temporada]
    jogos_vals = [jogos_por_temporada[t] for t in temporadas_presentes]

    ax2.bar(
        temporadas_presentes, jogos_vals,
        color='#1f77b4', alpha=0.4, edgecolor='#1f77b4'
    )
    for i, v in enumerate(jogos_vals):
        ax2.text(i, v + 5, str(v), ha='center', va='bottom', fontsize=8)

    ax2.set_ylabel('Jogos', fontsize=10)
    ax2.set_ylim(0, max(jogos_vals) * 1.25)
    ax2.grid(axis='y', linestyle='--', alpha=0.3)
    ax2.tick_params(axis='x', rotation=45)

    plt.tight_layout()
    plt.show()

    print(f'\nJanela = {janela} \u2014 Resumo Stack Ensemble (XGBoost):')
    print(df_stack[['Temporada', 'Jogos Avaliados', 'Acur\u00e1cia', 'F1-Score']].to_string(index=False))
    print()

## Visualizacao — Stack XGBoost vs Todos

Plotei o grafico comparativo para cada uma das 3 janelas incrementais (5, 10, 15).

Cada grafico mostra:
- As 5 linhas dos modelos individuais (bem transparentes)
- A linha do voto majoritario (cinza, pontilhada)
- A linha do soft voting (cinza, traco-ponto)
- A linha **azul** do stack ensemble com Regressao Logistica
- A linha **vermelha** do stack ensemble com XGBoost (destaque)

In [ ]:
for janela in JANELAS:
    plot_comparativo(janela)

## Resumo Comparativo Final

Aqui criei uma tabela final comparando a **media de acuracia** de todas as abordagens, incluindo agora o stack ensemble com XGBoost.

In [ ]:
for janela in JANELAS:
    print(f'\n{"=" * 75}')
    print(f'JANELA = {janela}')
    print(f'{"=" * 75}')

    tabela_comparativa = []

    modelos_individuais = {
        'SVM': 'svm', 'KNN': 'knn', 'Extra Trees': 'extra_trees',
        'Regressao Logistica': 'regressao_logistica', 'LightGBM': 'lightgbm'
    }
    for nome, arquivo in modelos_individuais.items():
        path = os.path.join(
            SAIDA_EXP03,
            f'{arquivo}_experimento_03_janela{janela}.csv'
        )
        if os.path.exists(path):
            df_m = pd.read_csv(path)
            tabela_comparativa.append({
                'Modelo': nome,
                'Media Acuracia': round(df_m['Acur\u00e1cia'].astype(float).mean(), 4),
                'Desvio Padrao': round(df_m['Acur\u00e1cia'].astype(float).std(), 4),
            })

    voto_path = os.path.join(SAIDA_EXP03, 'ensemble_voto_majoritario.csv')
    if os.path.exists(voto_path):
        df_v = pd.read_csv(voto_path)
        df_v = df_v[df_v['Janela Incremental'] == janela]
        tabela_comparativa.append({
            'Modelo': 'Voto Majoritario',
            'Media Acuracia': round(df_v['Acur\u00e1cia'].astype(float).mean(), 4),
            'Desvio Padrao': round(df_v['Acur\u00e1cia'].astype(float).std(), 4),
        })

    soft_path = os.path.join(SAIDA_EXP03, 'ensemble_soft_voting.csv')
    if os.path.exists(soft_path):
        df_s = pd.read_csv(soft_path)
        df_s = df_s[df_s['Janela Incremental'] == janela]
        tabela_comparativa.append({
            'Modelo': 'Soft Voting',
            'Media Acuracia': round(df_s['Acur\u00e1cia'].astype(float).mean(), 4),
            'Desvio Padrao': round(df_s['Acur\u00e1cia'].astype(float).std(), 4),
        })

    lr_path = os.path.join(SAIDA_STACK, 'stack_ensemble_resumo.csv')
    if os.path.exists(lr_path):
        df_lr = pd.read_csv(lr_path)
        df_lr = df_lr[df_lr['Janela Incremental'] == janela]
        tabela_comparativa.append({
            'Modelo': 'Stack (Log. Reg.)',
            'Media Acuracia': round(df_lr['Acur\u00e1cia'].astype(float).mean(), 4),
            'Desvio Padrao': round(df_lr['Acur\u00e1cia'].astype(float).std(), 4),
        })

    df_st = df_resumo_xgb[df_resumo_xgb['Janela Incremental'] == janela]
    tabela_comparativa.append({
        'Modelo': '>>> Stack (XGBoost) <<<',
        'Media Acuracia': round(df_st['Acur\u00e1cia'].astype(float).mean(), 4),
        'Desvio Padrao': round(df_st['Acur\u00e1cia'].astype(float).std(), 4),
    })

    df_comp = pd.DataFrame(tabela_comparativa)
    df_comp = df_comp.sort_values('Media Acuracia', ascending=False)
    print(df_comp.to_string(index=False))

## Diagnostico — Distribuicao das Previsoes vs Realidade

Para entender onde o modelo esta errando, comparei a distribuicao das **previsoes** com a distribuicao dos **resultados reais**.

In [ ]:
print('DISTRIBUICAO DAS PREVISOES vs REALIDADE (XGBoost)')
print('=' * 80)

for janela in JANELAS:
    df_j = df_predicoes[df_predicoes['Janela Incremental'] == janela]
    total = len(df_j)

    prev_1 = int((df_j['Previsao'] == 1).sum())
    prev_0 = int((df_j['Previsao'] == 0).sum())

    real_1 = int((df_j['Resultado Real'] == 1).sum())
    real_0 = int((df_j['Resultado Real'] == 0).sum())

    print(f'\nJanela={janela:>2}')
    print(f'  Previsoes:  total={total} | classe_1={prev_1} | classe_0={prev_0} | {prev_1}/{total} = {prev_1/total:.4f}')
    print(f'  Realidade:  total={total} | classe_1={real_1} | classe_0={real_0} | {real_1}/{total} = {real_1/total:.4f}')
    print(f'  Diferenca:  o modelo preve {abs(prev_1 - real_1)} jogos {"a mais" if prev_1 > real_1 else "a menos"} como classe 1')